# 05. FastAPI へのエクスポート（本番反映）

`data/staging/` の検証済み成果物を本番パスへプロモーションする。

⚠️ **このノートブックは本番データを変更します。**
`PROMOTE=True` に変更する前に必ず staging の内容を確認してください。

In [ ]:
# デフォルトは PROMOTE=False（確認モード）
# 本番反映する場合のみ True に変更
PROMOTE = False

import sys, json, logging
from pathlib import Path

try:
    BASE
except NameError:
    BASE        = Path("/content/AI_TradeManagement")
    STAGING_DIR = BASE / "data" / "staging"
    sys.path.insert(0, str(BASE / "scripts"))

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
print(f"PROMOTE={PROMOTE}  ({'🟢 本番反映モード' if PROMOTE else '🔵 確認モード (dry-run)'})") 

## 1. staging 成果物の確認

In [ ]:
from pipeline.export.staging_exporter import staging_summary

summary = staging_summary(STAGING_DIR)
print("staging ディレクトリの現在の成果物:")
print("-" * 60)
for path, info in summary.items():
    size = info.get("size_bytes", 0)
    rec  = f"  ({info['records']} 件)" if "records" in info else ""
    print(f"  {path:<45} {size:>10,} bytes{rec}")

## 2. プロモーション実行

In [ ]:
from pipeline.export.staging_exporter import promote

results = promote(staging_dir=STAGING_DIR, dry_run=not PROMOTE)
print("\nプロモーション結果:")
print("-" * 60)
for key, status in results.items():
    icon = "✅" if status == "promoted" else ("🔵" if "DRY_RUN" in status else ("⏭" if "skipped" in status else "❌"))
    print(f"  {icon} {key}: {status}")

## 3. 反映後のガイド

In [ ]:
if PROMOTE:
    print("✅ プロモーション完了")
    print("\n次のステップ:")
    print("  1. screening 再起動: POST http://localhost:8005 (FAISS リロード)")
    print("  2. matrix_rules FAISS リビルド: POST http://localhost:8001/api/admin/faiss/rebuild")
    print("  3. データセットリビルド: POST http://localhost:8000/admin/datasets/build")
else:
    print("🔵 確認モード完了。本番反映するには PROMOTE=True に変更して再実行してください。")